In [ ]:
#Modificar el bucle para que haga la llamada por 1 artista cada vez

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install requests

In [ ]:
import requests                # Importa el módulo requests para realizar peticiones HTTP.
import json                    # Importa el módulo json para manejar datos en formato JSON.

def search_artist_by_name(name, user_email):
    """ Busca un artista por nombre y retorna el primer resultado. """
    base_url = "https://musicbrainz.org/ws/2"  # URL base para el API de MusicBrainz.
    endpoint = "/artist/"                     # Endpoint específico para buscar artistas.
    params = {"query": name, "fmt": "json"}   # Parámetros para la búsqueda, incluyendo el formato de respuesta.
    headers = {"User-Agent": f"MiAplicacion/1.0 ({user_email})"}  # Headers necesarios para el request, incluyendo el User-Agent.

    # Realiza la petición HTTP GET con los parámetros y headers definidos.
    response = requests.get(base_url + endpoint, params=params, headers=headers)
    # Chequea si la respuesta fue exitosa.
    if response.status_code == 200:
        artists = response.json().get('artists', [])  # Extrae la lista de artistas de la respuesta JSON.
        if artists:  # Si hay artistas, retorna el primero.
            return artists[0]
    return None  # Retorna None si no hay artistas o si la petición falló.

def get_artist_details(mbid, user_email):
    """ Obtiene los detalles del artista, incluyendo sus grupos de lanzamiento y géneros. """
    base_url = "https://musicbrainz.org/ws/2"  # URL base para el API de MusicBrainz.
    endpoint = f"/release-group"              # Endpoint para obtener grupos de lanzamiento de un artista.
    params = {
        "artist": mbid,                       # ID de MusicBrainz del artista.
        "inc": "artist-credits+genres",       # Incluye créditos de artistas y géneros en la respuesta.
        "fmt": "json",                        # Formato de la respuesta.
        "limit": 100                          # Número máximo de resultados por consulta.
    }
    headers = {"User-Agent": f"MiAplicacion/1.0 ({user_email})"}  # Headers con User-Agent.

    # Realiza la petición HTTP GET con los parámetros y headers definidos.
    response = requests.get(base_url + endpoint, params=params, headers=headers)
    # Chequea si la respuesta fue exitosa.
    if response.status_code == 200:
        return response.json()  # Retorna los datos de la respuesta.
    return None  # Retorna None si la petición falló.

# Configuración principal para ejecutar el proceso
artists = ["Drake", "Beyoncé", "Rihanna", "Jay-Z", "Eminem", "Nicki Minaj", "David Guetta"]  # Lista de nombres de artistas a buscar.
user_email = "your_email@example.com"  # Email del usuario, necesario para los headers de las peticiones.
artist_data = []  # Lista para almacenar los datos recogidos de cada artista.

# Proceso iterativo para buscar cada artista y obtener sus detalles.
for artist_name in artists:
    artist = search_artist_by_name(artist_name, user_email)  # Busca al artista.
    if artist:
        print(f"Fetching details for {artist_name} with MBID: {artist['id']}")
        details = get_artist_details(artist['id'], user_email)  # Obtiene detalles del artista.
        artist_data.append(details if details else f"No data for {artist_name}")  # Añade los detalles o un mensaje de error a la lista.
    else:
        print(f"Artist {artist_name} not found.")  # Mensaje si el artista no se encuentra.

# Guarda los datos recolectados en un archivo JSON.
json_path = "/content/drive/MyDrive/artist_data.json"
with open(json_path, 'w') as f:
    json.dump(artist_data, f, ensure_ascii=False, indent=4)  # Escribe los datos en el archivo.

print("\nData saved to artist_data.json")  # Mensaje final indicando que los datos se guardaron correctamente.



Fetching details for Drake with MBID: 9fff2f8a-21e6-47de-a2b8-7f449929d43f
Fetching details for Beyoncé with MBID: 859d0860-d480-4efd-970c-c05d5f1776b8
Fetching details for Rihanna with MBID: 73e5e69d-3554-40d8-8516-00cb38737a1c
Fetching details for Jay-Z with MBID: f82bcf78-5b69-4622-a5ef-73800768d9ac
Fetching details for Eminem with MBID: b95ce3ff-3d05-4e87-9e01-c97b66af13d4
Fetching details for Nicki Minaj with MBID: 1036b808-f58c-4a3e-b461-a2c4492ecf1b
Fetching details for David Guetta with MBID: 302bd7b9-d012-4360-897a-93b00c855680


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/artist_data.json'

In [ ]:
import json                                 # Importa el módulo para manejar datos en formato JSON
import networkx as nx                       # Importa la biblioteca para la creación y manipulación de grafos complejos
import matplotlib.pyplot as plt             # Importa la biblioteca para la visualización de datos
import numpy as np                          # Importa la biblioteca para operaciones matemáticas avanzadas

def leer_datos_json(ruta_archivo):          # Función para leer datos desde un archivo JSON
    with open(ruta_archivo, 'r') as archivo: # Abre el archivo en modo lectura
        datos = json.load(archivo)          # Carga los datos del archivo JSON en una variable
    return datos                            # Devuelve los datos leídos del archivo

def extraer_informacion(datos):             # Función para extraer y procesar información de los datos JSON
    red_datos = []                          # Lista para almacenar datos procesados
    colaboradores_count = {}                # Diccionario para contar colaboradores por artista
    generos_unicos = set()                  # Conjunto para almacenar géneros únicos
    artistas_principales = ["Drake", "Beyoncé", "Rihanna", "Jay-Z", "Eminem", "Nicki Minaj", "David Guetta"] # Lista de artistas principales

    for entrada in datos:                   # Itera a través de cada entrada en los datos JSON
        for grupo in entrada.get('release-groups', []): # Itera a través de cada grupo de lanzamiento en la entrada
            for credito in grupo.get('artist-credit', []): # Itera a través de cada crédito de artista en el grupo
                artista_principal = credito.get('artist', {}).get('name') # Obtiene el nombre del artista principal
                if artista_principal in artistas_principales: # Verifica si el artista está en la lista de artistas principales
                    colaboradores = [cred['artist']['name'] for cred in grupo.get('artist-credit', []) if cred['artist']['name'] != artista_principal] # Lista de colaboradores
                    generos = [g['name'] for g in grupo.get('genres', [])] # Lista de géneros del grupo
                    generos_unicos.update(generos) # Agrega los géneros a un conjunto de géneros únicos
                    if artista_principal not in colaboradores_count:
                        colaboradores_count[artista_principal] = set() # Inicializa un conjunto si el artista no está ya en el diccionario
                    colaboradores_count[artista_principal].update(colaboradores) # Añade colaboradores al conjunto del artista principal
                    lanzamiento = { # Crea un diccionario con información del lanzamiento
                        'artista': artista_principal,
                        'titulo': grupo.get('title'),
                        'tipo': grupo.get('primary-type'),
                        'generos': generos,
                        'colaboradores': colaboradores
                    }
                    red_datos.append(lanzamiento) # Añade el diccionario a la lista de datos procesados
    return red_datos, colaboradores_count, list(generos_unicos) # Devuelve los datos procesados, el conteo de colaboradores y la lista de géneros únicos

def crear_grafo(datos, colaboradores_count, generos_unicos):
    G = nx.Graph()  # Crea un nuevo grafo
    color_map = {gen: plt.cm.turbo(i / (len(generos_unicos) - 1)) for i, gen in enumerate(generos_unicos)}  # Crea un mapa de colores para los géneros

    for dato in datos:  # Itera a través de cada dato procesado
        artista_principal = dato['artista']  # Obtiene el nombre del artista principal
        num_colaboradores = len(colaboradores_count[artista_principal])  # Obtiene el número de colaboradores
        G.add_node(artista_principal, type='artista_principal', size=5000, color='lightblue')  # Añade un nodo para el artista principal

        for genero in dato['generos']:  # Itera a través de los géneros del dato
            G.add_node(genero, type='genero', size=500, color=color_map[genero])  # Añade un nodo para cada género
            G.add_edge(artista_principal, genero, type='pertenece_a')  # Añade una arista entre el artista y el género

        for colaborador in dato['colaboradores']:  # Itera a través de los colaboradores del dato
            G.add_node(colaborador, type='colaborador', size=1000, color='yellow')  # Añade un nodo para cada colaborador
            G.add_edge(artista_principal, colaborador, type='colabora_con')  # Añade una arista entre el artista y el colaborador

        # Añade álbumes como nodos y los conecta
        album_title = dato['titulo'] if dato['titulo'] is not None else "Unknown Title"
        album_type = dato['tipo'] if dato['tipo'] is not None else "Unknown Type"
        album_id = f"{album_title} ({album_type})"  # Crea un identificador único para el álbum
        G.add_node(album_id, type='album', size=700, color='green')  # Añade un nodo para el álbum
        G.add_edge(artista_principal, album_id, type='pertenece_a_album')  # Añade una arista entre el artista y el álbum
        for colaborador in dato['colaboradores']:
            G.add_edge(colaborador, album_id, type='colabora_en_album')  # Añade aristas entre cada colaborador y el álbum

    pos = nx.spring_layout(G, scale=5)  # Calcula la posición de los nodos usando un layout de muelle

    # Ajustar las posiciones para diferentes tipos de nodos
    for node, data in G.nodes(data=True):
        if data['type'] in ['colaborador', 'album']:  # Ajustar nodos de tipo colaborador y álbum
            neighbors = list(G.neighbors(node))
            if len(neighbors) == 15:  # Considera los nodos con pocos vecinos#############################
                vecino = neighbors[0]
                vector = np.array(pos[node]) - np.array(pos[vecino])
                norm = np.linalg.norm(vector)
                if norm != 0:
                    vector /= norm  # Normaliza el vector para mantener la dirección
                    pos[node] = np.array(pos[vecino]) + vector * 5  # Ajusta el factor para modificar la distancia

    return G, color_map, pos  # Devuelve el grafo, el mapa de colores y las posiciones de los nodos


def main(): # Función principal que ejecuta todo el proceso
    ruta_archivo = '/content/drive/MyDrive/artist_data.json' # Define la ruta al archivo de datos
    datos_json = leer_datos_json(ruta_archivo) # Lee los datos del archivo JSON
    datos_estructurados, colaboradores_count, generos_unicos = extraer_informacion(datos_json) # Extrae información útil de los datos
    grafo, color_map, pos = crear_grafo(datos_estructurados, colaboradores_count, generos_unicos) # Crea un grafo con la información extraída

    if not nx.is_empty(grafo): # Verifica si el grafo no está vacío
        plt.figure(figsize=(100, 70)) # Establece el tamaño de la figura
        node_shapes = {
            'artista_principal': ('o', 600),  # Círculo, tamaño 200
            'genero': ('o', 100),             # Círculo, tamaño 100
            'colaborador': ('^', 150),        # Triángulo, tamaño 100
            'album': ('s', 150)               # Cuadrado, tamaño 150
        }

        for node_type, settings in node_shapes.items():
            shape, size = settings
            nodes = [node for node in grafo.nodes(data=True) if node[1]['type'] == node_type]
            nx.draw_networkx_nodes(
                grafo, pos,
                nodelist=[n[0] for n in nodes],
                node_size=size,
                node_color=[grafo.nodes[n[0]]['color'] for n in nodes],
                node_shape=shape,
                edgecolors='black'  # Añade un contorno negro a cada nodo
            )

        colors = [grafo.nodes[n]['color'] for n in grafo if 'color' in grafo.nodes[n]] # Obtiene los colores de los nodos
        sizes = [grafo.nodes[n]['size'] for n in grafo] # Obtiene los tamaños de los nodos
        labels = {node: node.replace('$', '\$') for node in grafo.nodes()}  # Escapa caracteres especiales en las etiquetas
        nx.draw_networkx_nodes(grafo, pos, node_color=colors, node_size=sizes) # Dibuja los nodos del grafo
        nx.draw_networkx_edges(grafo, pos, alpha=0.3) # Dibuja las aristas del grafo
        nx.draw_networkx_labels(grafo, pos, labels=labels) # Dibuja las etiquetas de los nodos
        plt.title('Red de Artistas, Géneros, Colaboradores y Álbumes') # Establece el título de la figura
        plt.figtext(0.5, 0.01, "\n".join([f"{a}: {len(c)}" for a, c in colaboradores_count.items()]), ha='center', fontsize=50, bbox={"facecolor":"orange", "alpha":0.5, "pad":5}) # Añade una leyenda para los colaboradores en la parte inferior central
        # Añade un título para la leyenda
        plt.figtext(0.5, 0.08, 'Artistas y el número de colaboradores', ha='center', fontsize=50, fontweight='bold')
        legend_handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=color_map[g], markersize=20, label=g) for g in generos_unicos] # Crea handles para la leyenda de géneros
        plt.legend(handles=legend_handles, loc='upper left', bbox_to_anchor=(1.5, 1), ncol=3) # Añade la leyenda de géneros en la parte superior derecha
        plt.axis('off') # Desactiva los ejes
        plt.show() # Muestra la figura
    else:
        print("El grafo está vacío.") # Imprime un mensaje si el grafo está vacío

main() # Ejecuta la función principal



Output hidden; open in https://colab.research.google.com to view.

In [ ]:
import json                                  # Importa el módulo json para la manipulación de datos JSON
import networkx as nx                        # Importa NetworkX para la manipulación y visualización de grafos
import matplotlib.pyplot as plt              # Importa matplotlib para la visualización gráfica
import numpy as np                           # Importa NumPy para cálculos numéricos

def leer_datos_json(ruta_archivo):           # Define una función para leer un archivo JSON
    with open(ruta_archivo, 'r') as archivo: # Abre el archivo en modo lectura
        datos = json.load(archivo)           # Carga los datos JSON del archivo
    return datos                             # Retorna los datos

def extraer_informacion(datos):              # Define una función para extraer información relevante de los datos
    red_datos = []                           # Inicializa una lista para almacenar los datos procesados
    colaboradores_count = {}                  # Diccionario para contar colaboradores por artista principal
    generos_unicos = set()                    # Conjunto para almacenar géneros únicos
    artistas_principales = ["Drake", "Beyoncé", "Rihanna", "Jay-Z", "Eminem", "Nicki Minaj", "David Guetta"] # Lista de artistas principales

    for entrada in datos:                     # Itera sobre cada entrada en los datos
        for grupo in entrada.get('release-groups', []): # Itera sobre cada grupo de lanzamiento
            for credito in grupo.get('artist-credit', []): # Itera sobre cada crédito de artista
                artista_principal = credito.get('artist', {}).get('name') # Obtiene el nombre del artista principal
                if artista_principal in artistas_principales: # Verifica si el artista está en la lista principal
                    colaboradores = [cred['artist']['name'] for cred in grupo.get('artist-credit', []) if cred['artist']['name'] != artista_principal] # Lista de colaboradores
                    generos = [g['name'] for g in grupo.get('genres', [])] # Lista de géneros del grupo
                    generos_unicos.update(generos) # Actualiza el conjunto de géneros únicos
                    if artista_principal not in colaboradores_count:
                        colaboradores_count[artista_principal] = set() # Inicializa el set de colaboradores si no existe
                    colaboradores_count[artista_principal].update(colaboradores) # Añade colaboradores al conjunto del artista
                    lanzamiento = { # Crea un diccionario con información del lanzamiento
                        'artista': artista_principal,
                        'titulo': grupo.get('title'),
                        'tipo': grupo.get('primary-type'),
                        'generos': generos,
                        'colaboradores': colaboradores
                    }
                    red_datos.append(lanzamiento) # Añade el diccionario a la lista de datos de la red
    return red_datos, colaboradores_count, list(generos_unicos) # Retorna los datos de la red, el conteo de colaboradores y la lista de géneros únicos

def crear_grafo(datos, colaboradores_count, generos_unicos): # Función para crear un grafo con los datos procesados
    G = nx.Graph() # Crea un nuevo grafo
    color_map = {gen: plt.cm.tab20(i / len(generos_unicos)) for i, gen in enumerate(generos_unicos)} # Mapa de colores para los géneros

    for dato in datos: # Itera sobre los datos procesados
        artista_principal = dato['artista'] # Obtiene el nombre del artista principal
        num_colaboradores = len(colaboradores_count[artista_principal]) # Número de colaboradores
        size = 100 + 10 * num_colaboradores # Calcula el tamaño del nodo basado en el número de colaboradores
        G.add_node(artista_principal, type='artista_principal', size=size, color='lightblue') # Añade el nodo del artista al grafo

        for genero in dato['generos']: # Itera sobre los géneros del artista
            G.add_node(genero, type='genero', size=100, color=color_map[genero]) # Añade el nodo del género al grafo
            G.add_edge(artista_principal, genero, type='pertenece_a') # Añade una arista entre el artista y el género

        for colaborador in dato['colaboradores']: # Itera sobre los colaboradores del artista
            G.add_node(colaborador, type='colaborador', size=100, color='gray') # Añade el nodo del colaborador al grafo
            G.add_edge(artista_principal, colaborador, type='colabora_con') # Añade una arista entre el artista y el colaborador

    pos = nx.spring_layout(G) # Usa el algoritmo de layout spring para calcular la posición de los nodos

    return G, color_map, pos # Retorna el grafo, el mapa de colores y la posición de los nodos

def main(): # Función principal que ejecuta todo el proceso
    ruta_archivo = '/content/drive/MyDrive/artist_data.json' # Ruta del archivo JSON
    datos_json = leer_datos_json(ruta_archivo) # Lee los datos del archivo JSON
    datos_estructurados, colaboradores_count, generos_unicos = extraer_informacion(datos_json) # Extrae la información necesaria de los datos
    grafo, color_map, pos = crear_grafo(datos_estructurados, colaboradores_count, generos_unicos) # Crea el grafo con los datos estructurados

    if not nx.is_empty(grafo): # Verifica si el grafo no está vacío
        plt.figure(figsize=(50, 35))  # Aumenta el tamaño de la figura para proporcionar más espacio para las leyendas
        colors = [grafo.nodes[n]['color'] for n in grafo if 'color' in grafo.nodes[n]] # Lista de colores para los nodos
        sizes = [grafo.nodes[n]['size'] for n in grafo] # Lista de tamaños para los nodos
        nx.draw_networkx_nodes(grafo, pos, node_color=colors, node_size=sizes) # Dibuja los nodos del grafo
        nx.draw_networkx_edges(grafo, pos, alpha=0.3) # Dibuja las aristas del grafo
        nx.draw_networkx_labels(grafo, pos) # Dibuja las etiquetas de los nodos del grafo
        plt.title('Red de Artistas, Géneros y Colaboradores') # Establece el título del gráfico

        plt.figtext(0.5, 0.01, "\n".join([f"{a}: {len(c)}" for a, c in colaboradores_count.items()]), ha='center', fontsize=9, bbox={"facecolor":"orange", "alpha":0.5, "pad":5}) # Añade una leyenda para los colaboradores en la parte inferior central

        legend_handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=color_map[g], markersize=10, label=g) for g in generos_unicos] # Crea handles para la leyenda de géneros
        plt.legend(handles=legend_handles, loc='upper left', bbox_to_anchor=(1.05, 1), ncol=4) # Añade la leyenda de géneros en la parte superior derecha

        plt.subplots_adjust(left=0.05, right=0.95, top=0.95, bottom=0.05) # Ajusta los márgenes para asegurar que todo el contenido sea visible
        plt.axis('off') # Desactiva los ejes
        plt.show() # Muestra el gráfico
    else:
        print("El grafo está vacío.") # Imprime un mensaje si el grafo está vacío

main() # Ejecuta la función principal


In [ ]:
import json
import networkx as nx
import matplotlib.pyplot as plt

# Cargar los datos desde el archivo JSON
with open('/content/drive/MyDrive/artist_data.json', 'r', encoding='utf-8') as file:
    data = json.load(file)

# Artistas principales para analizar
main_artists = ["Drake", "Beyoncé", "Rihanna", "Jay-Z", "Eminem", "Nicki Minaj", "David Guetta"]

# Crear el grafo
G = nx.Graph()

# Añadir los artistas como nodos y conectar cada álbum con su artista, omitiendo los de género 'Unknown'
for entry in data:
    if entry:
        artist_name = entry['release-groups'][0]['artist-credit'][0]['artist']['name']
        if artist_name in main_artists:
            G.add_node(artist_name, size=30, color='skyblue')  # Ajustar el tamaño base del nodo del artista
            for release_group in entry['release-groups']:
                if release_group['primary-type'] == 'Album':
                    # Tomar el primer género listado para el álbum, si está disponible
                    genre_label = release_group['genres'][0]['name'] if release_group['genres'] else 'Unknown'
                    if genre_label != 'Unknown':  # Sólo agregar si el género no es 'Unknown'
                        album_node = release_group['title']
                        G.add_node(album_node, size=10, color='lightgreen')  # Menor tamaño para nodos de álbumes
                        G.add_edge(artist_name, album_node, genre=genre_label)

# Ajustar el layout para incrementar la distancia entre nodos
pos = nx.spring_layout(G, k=0.5, seed=42)  # Aumentar k para más espacio entre los nodos

plt.figure(figsize=(14, 12))
node_sizes = [G.nodes[node]['size']*100 for node in G]
node_colors = [G.nodes[node]['color'] for node in G]
labels = nx.get_edge_attributes(G, 'genre')
nx.draw(G, pos, with_labels=True, node_size=node_sizes, node_color=node_colors, font_size=9, font_weight='bold', edge_color='gray')
nx.draw_networkx_edge_labels(G, pos, edge_labels=labels, font_color='red')
plt.title('Graph of Main Artists and Their Albums (Excluding Unknown Genres)')
plt.show()


In [ ]:
import json
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Cargar los datos desde el archivo JSON
with open('/content/drive/MyDrive/artist_data.json', 'r', encoding='utf-8') as file:
    data = json.load(file)

# Artistas principales para analizar
main_artists = ["Drake", "Beyoncé", "Rihanna", "Jay-Z", "Eminem", "Nicki Minaj", "David Guetta"]

# Seleccionar un conjunto de colores con alto contraste
high_contrast_colors = ['red', 'blue', 'green', 'orange', 'purple', 'brown', 'pink', 'olive', 'cyan', 'lime', 'gray', 'navy', 'teal', 'coral', 'gold']

# Crear el grafo
G = nx.Graph()

# Diccionario para mapear géneros a colores
genre_to_color = {}
color_index = 0

# Añadir los artistas como nodos y conectar cada álbum con su artista, omitiendo los de género 'Unknown'
for entry in data:
    if entry:
        artist_name = entry['release-groups'][0]['artist-credit'][0]['artist']['name']
        if artist_name in main_artists:
            G.add_node(artist_name, size=30, color='skyblue')  # Ajustar el tamaño base del nodo del artista
            for release_group in entry['release-groups']:
                if release_group['primary-type'] == 'Album':
                    genre_label = release_group['genres'][0]['name'] if release_group['genres'] else 'Unknown'
                    if genre_label != 'Unknown':
                        if genre_label not in genre_to_color:
                            genre_to_color[genre_label] = high_contrast_colors[color_index % len(high_contrast_colors)]
                            color_index += 1
                        album_node = release_group['title']
                        G.add_node(album_node, size=10, color=genre_to_color[genre_label])
                        G.add_edge(artist_name, album_node)

# Ajustar el layout para incrementar la distancia entre nodos
pos = nx.spring_layout(G, k=0.5, seed=42)  # Aumentar k para más espacio entre los nodos

plt.figure(figsize=(14, 12))
node_sizes = [G.nodes[node]['size']*100 for node in G]
node_colors = [G.nodes[node]['color'] for node in G]

# Dibujar el grafo con etiquetas solo para los artistas
nx.draw(G, pos, with_labels=False, node_size=node_sizes, node_color=node_colors, font_size=9, font_weight='bold', edge_color='gray')
# Dibujar solo etiquetas de los artistas principales
artist_labels = {node: node for node in G.nodes if node in main_artists}
nx.draw_networkx_labels(G, pos, labels=artist_labels, font_size=12, font_weight='bold')

# Añadir leyenda para los géneros
legend_handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=color, markersize=10, label=genre) for genre, color in genre_to_color.items()]
plt.legend(handles=legend_handles, title="Genres", bbox_to_anchor=(1.05, 1), loc='upper left')

plt.title('Graph of Main Artists and Their Albums (Colored by Genre)')
plt.show()


In [ ]:
import json
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Cargar los datos desde el archivo JSON
with open('/content/drive/MyDrive/artist_data.json', 'r', encoding='utf-8') as file:
    data = json.load(file)

# Artistas principales para analizar
main_artists = ["Drake", "Beyoncé", "Rihanna", "Jay-Z", "Eminem", "Nicki Minaj", "David Guetta"]

# Seleccionar un conjunto de colores con alto contraste
high_contrast_colors = ['red', 'blue', 'green', 'orange', 'purple', 'brown', 'pink', 'olive', 'cyan', 'lime', 'gray', 'navy', 'teal', 'coral', 'gold']

# Crear el grafo
G = nx.Graph()

# Diccionario para mapear géneros a colores y contar álbumes por artista
genre_to_color = {}
album_counts = {artist: 0 for artist in main_artists}
color_index = 0

# Añadir los artistas como nodos y conectar cada álbum con su artista, omitiendo los de género 'Unknown'
for entry in data:
    if entry:
        artist_name = entry['release-groups'][0]['artist-credit'][0]['artist']['name']
        if artist_name in main_artists:
            for release_group in entry['release-groups']:
                if release_group['primary-type'] == 'Album':
                    genre_label = release_group['genres'][0]['name'] if release_group['genres'] else 'Unknown'
                    if genre_label != 'Unknown':
                        if genre_label not in genre_to_color:
                            genre_to_color[genre_label] = high_contrast_colors[color_index % len(high_contrast_colors)]
                            color_index += 1
                        album_node = release_group['title']
                        G.add_node(album_node, size=10, color=genre_to_color[genre_label])
                        album_counts[artist_name] += 1

# Actualizar nodos de artistas con conteo de álbumes y asegurarse de que las aristas usen estos nuevos nodos
for artist in main_artists:
    artist_label = f"{artist} ({album_counts[artist]})"
    G.add_node(artist_label, size=30, color='skyblue')
    for album, data in G.nodes(data=True):
        if data.get('color') in high_contrast_colors:  # Esta es una forma de identificar los nodos de álbumes
            G.add_edge(artist_label, album)  # Conectar con el nuevo nodo de artista

# Ajustar el layout para incrementar la distancia entre nodos
pos = nx.spring_layout(G, k=0.5, seed=42)  # Aumentar k para más espacio entre los nodos

plt.figure(figsize=(14, 12))
node_sizes = [G.nodes[node]['size']*100 for node in G]
node_colors = [G.nodes[node]['color'] for node in G]

# Dibujar el grafo
nx.draw(G, pos, with_labels=True, node_size=node_sizes, node_color=node_colors, font_size=9, font_weight='bold', edge_color='gray')

# Añadir leyenda para los géneros
legend_handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=color, markersize=10, label=genre) for genre, color in genre_to_color.items()]
plt.legend(handles=legend_handles, title="Genres", bbox_to_anchor=(1.05, 1), loc='upper left')

plt.title('Graph of Main Artists and Their Albums (Colored by Genre)')
plt.show()



In [ ]:
import json
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Cargar los datos desde el archivo JSON
with open('/content/drive/MyDrive/artist_data.json', 'r', encoding='utf-8') as file:
    data = json.load(file)

# Artistas principales para analizar
main_artists = ["Drake", "Beyoncé", "Rihanna", "Jay-Z", "Eminem", "Nicki Minaj", "David Guetta"]

# Seleccionar un conjunto de colores con alto contraste
high_contrast_colors = ['red', 'blue', 'green', 'orange', 'purple', 'brown', 'pink', 'olive', 'cyan', 'lime', 'gray', 'navy', 'teal', 'coral', 'gold']

# Crear el grafo
G = nx.Graph()

# Diccionario para mapear géneros a colores y contar álbumes por artista
genre_to_color = {}
album_counts = {artist: 0 for artist in main_artists}
color_index = 0

# Añadir los artistas como nodos y conectar cada álbum con su artista, omitiendo los de género 'Unknown'
for entry in data:
    if entry:
        artist_name = entry['release-groups'][0]['artist-credit'][0]['artist']['name']
        if artist_name in main_artists:
            for release_group in entry['release-groups']:
                if release_group['primary-type'] == 'Album':
                    genre_label = release_group['genres'][0]['name'] if release_group['genres'] else 'Unknown'
                    if genre_label != 'Unknown':
                        if genre_label not in genre_to_color:
                            genre_to_color[genre_label] = high_contrast_colors[color_index % len(high_contrast_colors)]
                            color_index += 1
                        album_node = release_group['title']
                        G.add_node(album_node, size=10, color=genre_to_color[genre_label])
                        album_counts[artist_name] += 1

# Actualizar nodos de artistas con conteo de álbumes
for artist in main_artists:
    artist_label = f"{artist} ({album_counts[artist]})"
    G.add_node(artist_label, size=30, color='skyblue')
    for album, data in G.nodes(data=True):
        if data.get('color') in high_contrast_colors:  # Esta es una forma de identificar los nodos de álbumes
            G.add_edge(artist_label, album)  # Conectar con el nuevo nodo de artista

# Usar layout de spring con un parámetro modificado para los artistas principales
pos = nx.spring_layout(G, k=0.75, seed=42)  # Usar un valor más grande de k para más espacio

# Incrementar manualmente el espacio entre los nodos principales para mejorar la visualización
for artist in main_artists:
    artist_label = f"{artist} ({album_counts[artist]})"
    if artist_label in pos:
        # Mover los nodos de los artistas principales más lejos entre sí
        pos[artist_label] += 0.1 * pos[artist_label]

plt.figure(figsize=(14, 12))
node_sizes = [G.nodes[node]['size']*100 for node in G]
node_colors = [G.nodes[node]['color'] for node in G]

# Dibujar el grafo
nx.draw(G, pos, with_labels=True, node_size=node_sizes, node_color=node_colors, font_size=9, font_weight='bold', edge_color='gray')

# Añadir leyenda para los géneros
legend_handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=color, markersize=10, label=genre) for genre, color in genre_to_color.items()]
plt.legend(handles=legend_handles, title="Genres", bbox_to_anchor=(1.05, 1), loc='upper left')

plt.title('Graph of Main Artists and Their Albums (Colored by Genre)')
plt.show()


In [ ]:
import json
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

# Cargar los datos desde el archivo JSON
with open('/content/drive/MyDrive/artist_data.json', 'r', encoding='utf-8') as file:
    data = json.load(file)

# Artistas principales para analizar
main_artists = ["Drake", "Beyoncé", "Rihanna", "Jay-Z", "Eminem", "Nicki Minaj", "David Guetta"]

# Seleccionar un conjunto de colores con alto contraste
high_contrast_colors = ['red', 'blue', 'green', 'orange', 'purple', 'brown', 'pink', 'olive', 'cyan', 'lime', 'gray', 'navy', 'teal', 'coral', 'gold']

# Crear el grafo
G = nx.Graph()

# Diccionario para mapear géneros a colores y contar álbumes por artista
genre_to_color = {}
album_counts = {artist: 0 for artist in main_artists}
color_index = 0

# Añadir los artistas como nodos y conectar cada álbum con su artista, omitiendo los de género 'Unknown'
for entry in data:
    if entry:
        artist_name = entry['release-groups'][0]['artist-credit'][0]['artist']['name']
        if artist_name in main_artists:
            for release_group in entry['release-groups']:
                if release_group['primary-type'] == 'Album':
                    genre_label = release_group['genres'][0]['name'] if release_group['genres'] else 'Unknown'
                    if genre_label != 'Unknown':
                        if genre_label not in genre_to_color:
                            genre_to_color[genre_label] = high_contrast_colors[color_index % len(high_contrast_colors)]
                            color_index += 1
                        album_node = release_group['title']
                        G.add_node(album_node, size=10, color=genre_to_color[genre_label])
                        album_counts[artist_name] += 1

# Actualizar nodos de artistas con conteo de álbumes
for artist in main_artists:
    artist_label = f"{artist} ({album_counts[artist]})"
    G.add_node(artist_label, size=30, color='skyblue')
    for album, data in G.nodes(data=True):
        if data.get('color') in high_contrast_colors:
            G.add_edge(artist_label, album)

# Usar layout de spring con un parámetro modificado para los artistas principales
pos = nx.spring_layout(G, k=0.75, seed=42)

# Incrementar manualmente el espacio entre los nodos principales para mejorar la visualización
artist_positions = {artist: pos[f"{artist} ({album_counts[artist]})"] for artist in main_artists}
center = np.mean(list(artist_positions.values()), axis=0)
for artist, position in artist_positions.items():
    offset = position - center
    pos[artist] = position + 2 * offset  # Aumentar la separación multiplicando el desplazamiento

plt.figure(figsize=(14, 12))
node_sizes = [G.nodes[node]['size']*100 for node in G]
node_colors = [G.nodes[node]['color'] for node in G]

# Dibujar el grafo
nx.draw(G, pos, with_labels=True, node_size=node_sizes, node_color=node_colors, font_size=9, font_weight='bold', edge_color='gray')

# Añadir leyenda para los géneros
legend_handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=color, markersize=10, label=genre) for genre, color in genre_to_color.items()]
plt.legend(handles=legend_handles, title="Genres", bbox_to_anchor=(1.05, 1), loc='upper left')

plt.title('Graph of Main Artists and Their Albums (Colored by Genre)')
plt.show()


In [ ]:
import json
import networkx as nx
import matplotlib.pyplot as plt

# Load the data from the JSON file
with open('/content/drive/MyDrive/artist_data.json', 'r', encoding='utf-8') as file:
    data = json.load(file)

# Main artists to analyze
main_artists = ["Drake", "Beyoncé", "Rihanna", "Jay-Z", "Eminem", "Nicki Minaj", "David Guetta"]

# Create the graph
G = nx.Graph()

# Add main artists and their albums, skipping 'Unknown' genres
for entry in data:
    if entry:
        artist_credits = entry['release-groups'][0]['artist-credit']
        for credit in artist_credits:
            artist_name = credit['artist']['name']
            if artist_name in main_artists:
                G.add_node(artist_name, size=30, color='skyblue')  # Main artist node
                for release_group in entry['release-groups']:
                    if release_group['primary-type'] == 'Album':
                        genre_label = release_group['genres'][0]['name'] if release_group['genres'] else 'Unknown'
                        if genre_label != 'Unknown':
                            album_node = release_group['title']
                            G.add_node(album_node, size=10, color='lightgreen')  # Album node
                            G.add_edge(artist_name, album_node, genre=genre_label)
                            # Check if there are collaborating artists in the credit list
                            for sub_credit in artist_credits:
                                if sub_credit['artist']['name'] != artist_name:
                                    collab_artist = sub_credit['artist']['name']
                                    G.add_node(collab_artist, size=15, color='red')  # Collaborating artist node
                                    G.add_edge(collab_artist, album_node, label=artist_name)  # Connect collaborator to album

# Use kamada_kawai layout to reduce overlap
pos = nx.kamada_kawai_layout(G)

# Draw the graph
plt.figure(figsize=(14, 12))
node_sizes = [G.nodes[node]['size'] * 100 for node in G]
node_colors = [G.nodes[node]['color'] for node in G]
labels = nx.get_edge_attributes(G, 'genre')

# Draw graph with arc-shaped edges to avoid overlap
nx.draw(G, pos, with_labels=True, node_size=node_sizes, node_color=node_colors, font_size=9, font_weight='bold', edge_color='gray', connectionstyle='arc3,rad=0.2')
nx.draw_networkx_edge_labels(G, pos, edge_labels=labels, font_color='red')
plt.title('Graph of Main Artists, Their Albums, and Collaborators (Colored by Role)')
plt.show()


In [ ]:
import json
import networkx as nx
import matplotlib.pyplot as plt

# Cargar los datos desde el archivo JSON
with open('/content/drive/MyDrive/artist_data.json', 'r', encoding='utf-8') as file:
    data = json.load(file)

# Artistas principales para analizar
main_artists = ["Drake", "Beyoncé", "Rihanna", "Jay-Z", "Eminem", "Nicki Minaj", "David Guetta"]

# Crear el grafo
G = nx.Graph()

# Añadir los artistas como nodos y conectar cada álbum con su artista, omitiendo los de género 'Unknown'
for entry in data:
    if entry:
        artist_name = entry['release-groups'][0]['artist-credit'][0]['artist']['name']
        if artist_name in main_artists:
            G.add_node(artist_name, size=30, color='skyblue')  # Ajustar el tamaño base del nodo del artista
            for release_group in entry['release-groups']:
                if release_group['primary-type'] == 'Album':
                    # Tomar el primer género listado para el álbum, si está disponible
                    genre_label = release_group['genres'][0]['name'] if release_group['genres'] else 'Unknown'
                    if genre_label != 'Unknown':  # Sólo agregar si el género no es 'Unknown'
                        album_node = release_group['title']
                        G.add_node(album_node, size=10, color='lightgreen')  # Menor tamaño para nodos de álbumes
                        G.add_edge(artist_name, album_node, genre=genre_label)

# Identificar álbumes conectados con exactamente dos artistas principales
for node in G.nodes:
    if G.nodes[node].get('color') == 'lightgreen':  # Considerar solo nodos de álbumes
        connected_artists = [n for n in G.neighbors(node) if n in main_artists]
        if len(connected_artists) == 2:
            G.nodes[node]['color'] = 'red'  # Colorear de rojo los nodos de álbumes con dos conexiones

# Usar el layout kamada_kawai para reducir solapamiento
pos = nx.kamada_kawai_layout(G)

# Dibujar el grafo
plt.figure(figsize=(14, 12))
node_sizes = [G.nodes[node]['size'] * 100 for node in G]
node_colors = [G.nodes[node]['color'] for node in G]
labels = nx.get_edge_attributes(G, 'genre')

# Dibujar el grafo con curvatura en las aristas para evitar solapamiento
nx.draw(G, pos, with_labels=True, node_size=node_sizes, node_color=node_colors, font_size=9, font_weight='bold', edge_color='gray', connectionstyle='arc3,rad=0.2')
nx.draw_networkx_edge_labels(G, pos, edge_labels=labels, font_color='red')
plt.title('Graph of Main Artists and Their Albums (Colored by Genre)')
plt.show()


In [ ]:
import json
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np

def escape_latex_string(text):
    return text.replace('$', r'\$').replace('_', r'\_').replace('%', r'\%')

# Cargar datos desde el archivo JSON
json_path = "/content/drive/MyDrive/artist_data.json"
with open(json_path, 'r') as file:
    data = json.load(file)

G = nx.Graph()
album_count = {}
pos = {}

main_artists = ["Drake"]

# Definir radios para los círculos concéntricos
radius_main = 1.0  # Radio para artistas principales
radius_album = 3.0  # Aumento del radio para álbumes
radius_collaborator = 4.5  # Aumento del radio para colaboradores

# Añadir nodos y aristas al grafo
angle_main = 0  # Ángulo inicial para artistas principales
album_angles = []  # Lista para guardar los ángulos de los álbumes

for artist_info in data:
    if "release-groups" not in artist_info:
        continue
    num_albums = len([rg for rg in artist_info['release-groups'] if 'artist-credit' in rg])
    angle_increment_album = 2 * np.pi / num_albums if num_albums else 0
    angle_album = 0  # Inicializar el ángulo para los álbumes

    for release_group in artist_info['release-groups']:
        if 'artist-credit' not in release_group:
            continue
        for artist_credit in release_group['artist-credit']:
            artist_name = escape_latex_string(artist_credit['artist']['name'])
            artist_id = artist_credit['artist']['id']
            if artist_name in main_artists:
                if artist_id not in album_count:
                    album_count[artist_id] = 0
                album_count[artist_id] += 1
                if artist_id not in pos:
                    G.add_node(artist_id, label=artist_name, type='main_artist', color='red')
                    pos[artist_id] = np.array([np.cos(angle_main), np.sin(angle_main)]) * radius_main

                album_title = escape_latex_string(release_group['title'])
                album_id = release_group['id']
                G.add_node(album_id, label=album_title, type='album', color='lightblue')
                G.add_edge(artist_id, album_id)
                pos[album_id] = pos[artist_id] + np.array([np.cos(angle_album), np.sin(angle_album)]) * radius_album
                angle_album += angle_increment_album  # Incrementar el ángulo para el próximo álbum

                if 'artist-credit' in release_group:
                    for sub_artist in release_group['artist-credit']:
                        collaborator_name = escape_latex_string(sub_artist['artist']['name'])
                        collaborator_id = sub_artist['artist']['id']
                        if collaborator_id != artist_id:
                            unique_collaborator_id = f"{collaborator_id}_{album_id}"
                            G.add_node(unique_collaborator_id, label=collaborator_name, type='collaborator', color='lightgreen')
                            G.add_edge(album_id, unique_collaborator_id)
                            angle_collaborator = np.random.uniform(-0.5, 0.5) + angle_album  # Pequeña variación alrededor del ángulo del álbum
                            pos[unique_collaborator_id] = pos[album_id] + np.array([np.cos(angle_collaborator), np.sin(angle_collaborator)]) * radius_collaborator

# Dibujar el grafo
plt.figure(figsize=(12, 12))
nx.draw(G, pos, labels={n: G.nodes[n]['label'] for n in G.nodes()}, with_labels=True, node_color=[G.nodes[n]['color'] for n in G], node_size=[album_count.get(n, 1) * 100 if G.nodes[n]['type'] == 'main_artist' else 300 if G.nodes[n]['type'] == 'album' else 100 for n in G], font_size=8, font_weight='bold')
plt.title("Visualización de Red de Artistas y Colaboraciones")
plt.show()


In [ ]:
import json
import networkx as nx
import matplotlib.pyplot as plt

def escape_latex_string(text):
    # Escapa los caracteres que podrían ser interpretados como comandos de LaTeX
    return text.replace('$', r'\$').replace('_', r'\_').replace('%', r'\%')

# Cargamos los datos desde el archivo JSON
json_path = "/content/drive/MyDrive/artist_data.json"
with open(json_path, 'r') as file:
    data = json.load(file)

G = nx.Graph()
album_count = {}  # Diccionario para contar los álbumes de cada artista principal

# Lista de artistas principales a colorear en rojo
main_artists = ["Drake", "Beyoncé", "Rihanna", "Jay-Z", "Eminem", "Nicki Minaj", "David Guetta"]

# Añadir nodos y aristas al grafo
for artist_info in data:
    if "release-groups" not in artist_info:
        continue
    for release_group in artist_info['release-groups']:
        if 'artist-credit' not in release_group:
            continue
        for artist_credit in release_group['artist-credit']:
            main_artist_name = escape_latex_string(artist_credit['artist']['name'])
            main_artist_id = artist_credit['artist']['id']
            if main_artist_id not in album_count:
                album_count[main_artist_id] = 0
            album_count[main_artist_id] += 1  # Contar los álbumes para cada artista principal

            if main_artist_name in main_artists:
                G.add_node(main_artist_id, label=main_artist_name, type='main_artist', color='red')

            album_title = escape_latex_string(release_group['title'])
            album_id = release_group['id']
            G.add_node(album_id, label=album_title, type='album', color='blue')
            if main_artist_name in main_artists:
                G.add_edge(main_artist_id, album_id)  # Enlace artista principal a álbum

            # Añadir colaboraciones, creando un nodo único por colaboración por álbum
            if 'artist-credit' in release_group:
                for sub_artist in release_group['artist-credit']:
                    collaborator_name = escape_latex_string(sub_artist['artist']['name'])
                    collaborator_id = sub_artist['artist']['id']
                    if collaborator_id != main_artist_id and main_artist_name in main_artists:
                        unique_collaborator_id = f"{collaborator_id}_{album_id}"  # Crear un identificador único por álbum
                        G.add_node(unique_collaborator_id, label=collaborator_name, type='collaborator', color='green')
                        G.add_edge(album_id, unique_collaborator_id)  # Enlace único álbum a colaborador

# Dibujar el grafo
plt.figure(figsize=(12, 12))
# Posicionamiento de los nodos utilizando un layout ajustado por tipos de nodos
pos = nx.spring_layout(G, k=0.75, iterations=50)
for node in G:
    if G.nodes[node]['type'] == 'album':
        # Aumentar el espacio alrededor de los nodos de álbumes ajustando sus posiciones
        connected_nodes = list(G.neighbors(node))
        if connected_nodes:
            pos[node] = [pos[node][0] * 1.2, pos[node][1] * 1.2]  # Ajustar posiciones x, y para expandir

# Definir colores y tamaños para los diferentes tipos de nodos
colors = [G.nodes[n]['color'] for n in G]  # Usar el color definido en cada nodo
sizes = [album_count.get(n, 1) * 100 if G.nodes[n]['type'] == 'main_artist' else 300 if G.nodes[n]['type'] == 'album' else 100 for n in G]

nx.draw(G, pos, labels={n: G.nodes[n]['label'] for n in G.nodes()}, with_labels=True, node_color=colors, node_size=sizes, font_size=8, font_weight='bold')
plt.title("Visualización de Red de Artistas y Colaboraciones")
plt.show()


In [ ]:
import json
import networkx as nx
import matplotlib.pyplot as plt

def escape_latex_string(text):
    # Escapa los caracteres que podrían ser interpretados como comandos de LaTeX
    return text.replace('$', r'\$').replace('_', r'\_').replace('%', r'\%')

# Cargamos los datos desde el archivo JSON
json_path = "/content/drive/MyDrive/artist_data.json"
with open(json_path, 'r') as file:
    data = json.load(file)

G = nx.Graph()
album_count = {}  # Diccionario para contar los álbumes de cada artista principal

# Lista de artistas principales a colorear en rojo
main_artists = ["Drake", "Beyoncé", "Rihanna", "Jay-Z", "Eminem", "Nicki Minaj", "David Guetta"]

# Añadir nodos y aristas al grafo
for artist_info in data:
    if "release-groups" not in artist_info:
        continue
    for release_group in artist_info['release-groups']:
        if 'artist-credit' not in release_group:
            continue
        for artist_credit in release_group['artist-credit']:
            main_artist_name = escape_latex_string(artist_credit['artist']['name'])
            main_artist_id = artist_credit['artist']['id']
            if main_artist_id not in album_count:
                album_count[main_artist_id] = 0
            album_count[main_artist_id] += 1  # Contar los álbumes para cada artista principal

            # Verificar si el artista es uno de los principales para colorear en rojo
            if main_artist_name in main_artists:
                G.add_node(main_artist_id, label=main_artist_name, type='main_artist', color='red')
                node_type_main_artist = 'main_artist'
            else:
                node_type_main_artist = 'collaborator'  # Otros artistas como colaboradores (no deberían tener nodos rojos)

            album_title = escape_latex_string(release_group['title'])
            album_id = release_group['id']
            G.add_node(album_id, label=album_title, type='album', color='blue')
            if node_type_main_artist == 'main_artist':
                G.add_edge(main_artist_id, album_id)  # Enlace artista principal a álbum

            # Añadir colaboraciones
            if 'artist-credit' in release_group:
                for sub_artist in release_group['artist-credit']:
                    collaborator_name = escape_latex_string(sub_artist['artist']['name'])
                    collaborator_id = sub_artist['artist']['id']
                    if collaborator_id != main_artist_id:
                        G.add_node(collaborator_id, label=collaborator_name, type='collaborator', color='green')
                        G.add_edge(album_id, collaborator_id)  # Enlace álbum a colaborador (verde con azul)

# Dibujar el grafo
plt.figure(figsize=(12, 12))
# Posicionamiento de los nodos utilizando un layout ajustado por tipos de nodos
pos = nx.spring_layout(G, k=0.75, iterations=50)
for node in G:
    if G.nodes[node]['type'] == 'album':
        # Aumentar el espacio alrededor de los nodos de álbumes ajustando sus posiciones
        connected_nodes = list(G.neighbors(node))
        if connected_nodes:
            pos[node] = [pos[node][0] * 1.2, pos[node][1] * 1.2]  # Ajustar posiciones x, y para expandir

# Definir colores y tamaños para los diferentes tipos de nodos
colors = [G.nodes[n]['color'] for n in G]  # Usar el color definido en cada nodo
sizes = [album_count.get(n, 1) * 100 if G.nodes[n]['type'] == 'main_artist' else 300 if G.nodes[n]['type'] == 'album' else 100 for n in G]

nx.draw(G, pos, labels={n: G.nodes[n]['label'] for n in G.nodes()}, with_labels=True, node_color=colors, node_size=sizes, font_size=8, font_weight='bold')
plt.title("Visualización de Red de Artistas y Colaboraciones")
plt.show()


In [ ]:
import json
import networkx as nx
import matplotlib.pyplot as plt

def escape_latex_string(text):
    # Escapa los caracteres que podrían ser interpretados como comandos de LaTeX
    return text.replace('$', r'\$').replace('_', r'\_').replace('%', r'\%')

# Cargamos los datos desde el archivo JSON
json_path = "/content/drive/MyDrive/artist_data.json"
with open(json_path, 'r') as file:
    data = json.load(file)

G = nx.Graph()
album_count = {}  # Diccionario para contar los álbumes de cada artista principal

# Lista de artistas principales a colorear en rojo
main_artists = ["Drake", "Beyoncé", "Rihanna", "Jay-Z", "Eminem", "Nicki Minaj", "David Guetta"]

# Añadir nodos y aristas al grafo
for artist_info in data:
    if "release-groups" not in artist_info:
        continue
    for release_group in artist_info['release-groups']:
        if 'artist-credit' not in release_group:
            continue
        for artist_credit in release_group['artist-credit']:
            main_artist_name = escape_latex_string(artist_credit['artist']['name'])
            main_artist_id = artist_credit['artist']['id']
            if main_artist_id not in album_count:
                album_count[main_artist_id] = 0
            album_count[main_artist_id] += 1  # Contar los álbumes para cada artista principal

            # Verificar si el artista es uno de los principales para colorear en rojo
            if main_artist_name in main_artists:
                G.add_node(main_artist_id, label=main_artist_name, type='main_artist', color='red')
            else:
                G.add_node(main_artist_id, label=main_artist_name, type='collaborator', color='green')

            album_title = escape_latex_string(release_group['title'])
            album_id = release_group['id']
            G.add_node(album_id, label=album_title, type='album', color='blue')
            G.add_edge(main_artist_id, album_id)  # Enlace artista principal a álbum

            # Añadir colaboraciones
            if 'artist-credit' in release_group:
                for sub_artist in release_group['artist-credit']:
                    collaborator_name = escape_latex_string(sub_artist['artist']['name'])
                    collaborator_id = sub_artist['artist']['id']
                    if collaborator_id != main_artist_id:
                        G.add_node(collaborator_id, label=collaborator_name, type='collaborator', color='green')
                        G.add_edge(album_id, collaborator_id)  # Enlace álbum a colaborador

# Dibujar el grafo
plt.figure(figsize=(12, 12))
# Posicionamiento de los nodos utilizando un layout ajustado por tipos de nodos
pos = nx.spring_layout(G, k=0.75, iterations=50)
for node in G:
    if G.nodes[node]['type'] == 'album':
        # Aumentar el espacio alrededor de los nodos de álbumes ajustando sus posiciones
        connected_nodes = list(G.neighbors(node))
        if connected_nodes:
            pos[node] = [pos[node][0] * 1.2, pos[node][1] * 1.2]  # Ajustar posiciones x, y para expandir

# Definir colores y tamaños para los diferentes tipos de nodos
colors = [G.nodes[n]['color'] for n in G]  # Usar el color definido en cada nodo
sizes = [album_count.get(n, 1) * 100 if G.nodes[n]['type'] == 'main_artist' else 300 if G.nodes[n]['type'] == 'album' else 100 for n in G]

nx.draw(G, pos, labels={n: G.nodes[n]['label'] for n in G.nodes()}, with_labels=True, node_color=colors, node_size=sizes, font_size=8, font_weight='bold')
plt.title("Visualización de Red de Artistas y Colaboraciones")
plt.show()


Pregunta 1: Red de colaboración: ¿Qué artistas colaboran frecuentemente con otros? Al trazar un mapa de las colaboraciones, podemos identificar a influyentes clave y figuras centrales de la industria musical que tienden puentes entre distintos géneros musicales o comunidades. ##### Probar con grafo para cada artista seleccionado individualmente


Pregunta 2: Colaboraciones en álbumes: ¿Cómo se interconectan los artistas a través de los distintos álbumes? En el caso de los álbumes en los que aparecen varios artistas, comprender estas conexiones puede revelar la naturaleza colaborativa de varios proyectos y cómo ciertos álbumes sirven de centros de intersección artística.

Pregunta 3: Cruce de géneros: ¿Qué artistas tienden puentes entre distintos géneros musicales a través de sus colaboraciones? Esto puede analizarse clasificando el género principal de cada artista y examinando sus colaboraciones entre géneros.


Pregunta 4:
Análisis temporal de las colaboraciones: ¿Cómo han evolucionado las colaboraciones a lo largo del tiempo? El seguimiento de las colaboraciones a lo largo de los años puede ayudar a identificar tendencias, como el aumento o la disminución de la colaboración entre artistas, la aparición de nuevos grupos de colaboración o la persistencia de parejas/grupos de colaboración.
